# TP2 — Stacked LSTM — Prédiction de la Consommation Électrique
## Dataset : Household Electric Power Consumption (UCI)
**Étudiant :** AYOUB BARHOINE | **Filière :** Cycle Ingénieur GI 2ème Année | **Module :** Deep Learning | **Année :** 2026

## Table des Matières
1. Contexte et Dataset
2. Import des bibliothèques
3. Chargement et nettoyage des données
4. Feature Engineering
5. Sélection des 11 features multivariées
6. Normalisation et création des séquences
7. Découpage Train/Test + DataLoader
8. Architecture Stacked LSTM (3 couches)
9. Entraînement (50 epochs)
10. Prédictions et visualisation
11. Évaluation (MAE, RMSE, R2)
12. Conclusion

## 1. Contexte et Dataset

Le dataset **Household Electric Power Consumption** (UCI) contient des mesures minute par minute de la consommation électrique d'un foyer français sur 4 ans (2006-2010). Après nettoyage il contient **2 049 280 lignes**.

- **Cible :** `Global_active_power` (Puissance active en kW)
- **Features :** 7 mesures physiques + 4 features temporelles
- **Source UCI :** https://archive.ics.uci.edu/ml/datasets/Individual+household+electric+power+consumption

## 2. Import des bibliothèques

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import warnings
warnings.filterwarnings('ignore')

# Reproductibilité
torch.manual_seed(42)
np.random.seed(42)

# Device (GPU si disponible, sinon CPU)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device utilisé : {device}')
print('✅ Bibliothèques importées avec succès')

## 3. Chargement et nettoyage des données

In [ ]:
# --- Téléchargement automatique depuis UCI ---
import urllib.request, zipfile, os

URL  = 'https://archive.ics.uci.edu/ml/machine-learning-databases/00235/household_power_consumption.zip'
FILE = 'household_power_consumption.zip'
TXT  = 'household_power_consumption.txt'

if not os.path.exists(TXT):
    print('Téléchargement du dataset...')
    urllib.request.urlretrieve(URL, FILE)
    with zipfile.ZipFile(FILE, 'r') as z:
        z.extractall('.')
    print('✅ Dataset téléchargé.')
else:
    print('✅ Dataset déjà présent.')

# Chargement
df = pd.read_csv(
    TXT, sep=';', low_memory=False,
    parse_dates={'datetime': ['Date', 'Time']},
    dayfirst=True, na_values='?'
)

# Nettoyage — suppression NaN
df.dropna(inplace=True)
df.set_index('datetime', inplace=True)

print(f'✅ Dataset chargé : {len(df):,} lignes')
print(f'Période : du {df.index.min()} au {df.index.max()}')
display(df.head())

## 4. Feature Engineering

In [ ]:
# Extraction des features temporelles
df['hour']       = df.index.hour
df['dayofweek']  = df.index.dayofweek
df['month']      = df.index.month
df['is_weekend'] = (df.index.dayofweek >= 5).astype(int)

# Encodage cyclique de l'heure
# Permet au LSTM de comprendre que 23h et 0h sont temporellement proches
df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)

print('✅ Features temporelles ajoutées')
print(f'Exemple heure 17h : hour_sin = {df[df.hour==17].hour_sin.iloc[0]:.4f}, hour_cos = {df[df.hour==17].hour_cos.iloc[0]:.4f}')
display(df[['Global_active_power', 'hour', 'hour_sin', 'hour_cos', 'is_weekend']].head())

## 5. Sélection des 11 features multivariées

In [ ]:
feature_columns = [
    'Global_active_power',    # ← cible principale (index 0)
    'Global_reactive_power',
    'Voltage',
    'Global_intensity',
    'Sub_metering_1',
    'Sub_metering_2',
    'Sub_metering_3',
    'hour',
    'hour_sin',
    'hour_cos',
    'is_weekend'
]

dataset = df[feature_columns].values
print(f'✅ {len(feature_columns)} features sélectionnées : {feature_columns}')
print(f'Dataset numpy shape : {dataset.shape}')

## 6. Normalisation et création des séquences

In [ ]:
# Normalisation MinMax [0, 1]
scaler        = MinMaxScaler()
target_scaler = MinMaxScaler()

data_scaled = scaler.fit_transform(dataset)
target_scaler.fit(dataset[:, 0:1])  # scaler séparé pour dénormaliser les prédictions

# Création des séquences glissantes
SEQ_LENGTH = 60  # 60 minutes en entrée → prédire t+1

def create_sequences(data, seq_len):
    X, y = [], []
    for i in range(len(data) - seq_len):
        X.append(data[i:i+seq_len])       # fenêtre de 60 pas
        y.append(data[i+seq_len, 0])      # Global_active_power au t+1
    return np.array(X), np.array(y)

X, y = create_sequences(data_scaled, SEQ_LENGTH)

print(f'✅ Séquences créées : {len(X):,} exemples')
print(f'Shape X : {X.shape} | Shape y : {y.shape}')

## 7. Découpage Train/Test + DataLoader

In [ ]:
# Découpage chronologique 80/20
split = int(0.8 * len(X))
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

print(f'Train : {len(X_train):,} | Test : {len(X_test):,}')

# Conversion en tenseurs
X_train_t = torch.FloatTensor(X_train).to(device)
X_test_t  = torch.FloatTensor(X_test).to(device)
y_train_t = torch.FloatTensor(y_train).unsqueeze(1).to(device)
y_test_t  = torch.FloatTensor(y_test).unsqueeze(1).to(device)

print(f'X_train tensor shape : {X_train_t.shape}')
print(f'y_train tensor shape : {y_train_t.shape}')

# DataLoader
BATCH_SIZE = 64
train_dataset = TensorDataset(X_train_t, y_train_t)
train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=False)
print(f'Nb batches/epoch : {len(train_loader):,}')

## 8. Architecture Stacked LSTM (3 couches)

In [ ]:
class StackedLSTM(nn.Module):
    """3 couches LSTM empilées (128 unités, dropout=0.3) + couche Dense(1)"""
    def __init__(self, input_size=11, hidden_size=128, num_layers=3, dropout=0.3):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout
        )
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.lstm(x)   # out: (batch, seq_len, hidden_size)
        return self.fc(out[:, -1, :])  # dernière sortie temporelle

model = StackedLSTM(
    input_size=len(feature_columns),
    hidden_size=128,
    num_layers=3,
    dropout=0.3
).to(device)

print(f'✅ Modèle Stacked LSTM (3 couches) créé sur {device}')
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Paramètres entraînables : {total_params:,}')
print(model)

## 9. Entraînement (50 epochs)

In [ ]:
# Entraînement
N_EPOCHS  = 50
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

epoch_losses = []
checkpoints  = [1, 10, 20, 30, 40, 50]  # epochs à afficher

for epoch in range(1, N_EPOCHS + 1):
    model.train()
    epoch_loss = 0.0
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        y_pred = model(X_batch)
        loss   = criterion(y_pred, y_batch)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

    avg_loss = epoch_loss / len(train_loader)
    epoch_losses.append(avg_loss)

    if epoch in checkpoints:
        print(f'Epoch {epoch:3d}/{N_EPOCHS} — MSE Loss : {avg_loss:.4f}')

print('\n✅ Entraînement terminé')

In [ ]:
# Courbe de perte
plt.figure(figsize=(12, 5))
plt.plot(range(1, N_EPOCHS+1), epoch_losses, color='steelblue', linewidth=2)
# Annotations des checkpoints
for ep in checkpoints:
    plt.annotate(f'Epoch {ep}\n{epoch_losses[ep-1]:.4f}',
                 xy=(ep, epoch_losses[ep-1]),
                 xytext=(ep+1, epoch_losses[ep-1]+0.002),
                 fontsize=8, color='red',
                 arrowprops=dict(arrowstyle='->', color='red', lw=1.5))
    plt.scatter(ep, epoch_losses[ep-1], color='red', zorder=5, s=60)
plt.title('Courbe de perte pendant l\'entraînement (50 epochs)', fontsize=14)
plt.xlabel('Epoch'); plt.ylabel('MSE Loss')
plt.legend(['Training Loss (MSE)'])
plt.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 10. Prédictions et visualisation

In [ ]:
# Prédictions sur l'ensemble de test
model.eval()
with torch.no_grad():
    y_pred_scaled = model(X_test_t).cpu().numpy()

# Dénormalisation
y_pred_real = target_scaler.inverse_transform(y_pred_scaled)
y_true_real = target_scaler.inverse_transform(y_test.reshape(-1, 1))

# Visualisation — 500 premiers pas de temps
N_PLOT = 500
plt.figure(figsize=(14, 6))
plt.plot(y_true_real[:N_PLOT], label='Consommation réelle',     color='steelblue', linewidth=1.5)
plt.plot(y_pred_real[:N_PLOT], label='Prédiction Stacked LSTM', color='red',       linewidth=1, alpha=0.8)
plt.title('Prédiction de la consommation électrique (Stacked LSTM 3 couches)', fontsize=13)
plt.xlabel('Pas de temps dans l\'ensemble de test')
plt.ylabel('Global Active Power (kW)')
plt.legend(); plt.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 11. Évaluation — MAE, RMSE, R²

In [ ]:
mae  = mean_absolute_error(y_true_real, y_pred_real)
rmse = np.sqrt(mean_squared_error(y_true_real, y_pred_real))
r2   = r2_score(y_true_real, y_pred_real)

print('=' * 40)
print('       RÉSULTATS DU MODÈLE')
print('=' * 40)
print(f'Mean Absolute Error  (MAE) : {mae:.4f} kW')
print(f'Root Mean Squared Error    : {rmse:.4f} kW')
print(f'R² Score                   : {r2:.4f}')
print('=' * 40)

print('\nInterprétation :')
print(f'  • MAE {mae:.4f} kW = erreur moyenne de {mae*1000:.0f} W (consommation varie entre 0.2 et 11 kW)')
print(f'  • R² {r2:.4f} = {r2*100:.1f}% de la variance expliquée — excellent')

## 12. Conclusion

In [ ]:
summary = {
    'Aspect': ['Dataset', 'Features', 'Séquences', 'Architecture', 'Paramètres',
               'Optimisation', 'Entraînement', 'MAE / RMSE / R²'],
    'Détail': [
        '2 049 280 lignes — Household Power UCI (2006-2010)',
        '11 features (7 physiques + 4 temporelles)',
        'SEQ_LENGTH=60, fenêtre glissante → 2M exemples',
        'Stacked LSTM : 3 couches × 128 unités + Dropout(0.3)',
        f'{sum(p.numel() for p in model.parameters() if p.requires_grad):,} paramètres entraînables',
        'Adam (lr=0.001) + MSELoss + DataLoader(batch=64)',
        '50 epochs — convergence à partir de epoch 10',
        f'{mae:.4f} kW / {rmse:.4f} kW / {r2:.4f}'
    ]
}
print(pd.DataFrame(summary).to_string(index=False))
print('\nAYOUB BARHOINE — FST Settat — Deep Learning — Cycle Ingénieur GI 2ème année — 2026')